# Benchmarks

## Initialize

In [1]:
import os
import math
import pathlib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
from tqdm.auto import tqdm
from IPython.display import clear_output

import warnings
from lifelines.utils import CensoringType
from lifelines.utils import concordance_index

In [2]:
import ray
ray.shutdown()

In [3]:
node = !hostname
if "sc" in node[0]:
    base_path = "/sc-projects/sc-proj-ukb-cvd"
else: 
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"
print(base_path)

project_label = "22_medical_records"
project_path = f"{base_path}/results/projects/{project_label}"
figure_path = f"{project_path}/figures"
output_path = f"{project_path}/data"

pathlib.Path(figure_path).mkdir(parents=True, exist_ok=True)
pathlib.Path(output_path).mkdir(parents=True, exist_ok=True)

experiment = 220413
experiment_path = f"{output_path}/{experiment}"
pathlib.Path(experiment_path).mkdir(parents=True, exist_ok=True)

/sc-projects/sc-proj-ukb-cvd


In [4]:
import ray
ray.init(num_cpus=24, include_dashboard=False)#, dashboard_port=24762, dashboard_host="0.0.0.0", include_dashboard=True)#, webui_url="0.0.0.0"))

{'node_ip_address': '10.32.105.6',
 'raylet_ip_address': '10.32.105.6',
 'redis_address': None,
 'object_store_address': '/tmp/ray/session_2022-04-15_07-48-44_176469_2348875/sockets/plasma_store',
 'raylet_socket_name': '/tmp/ray/session_2022-04-15_07-48-44_176469_2348875/sockets/raylet',
 'webui_url': None,
 'session_dir': '/tmp/ray/session_2022-04-15_07-48-44_176469_2348875',
 'metrics_export_port': 61282,
 'gcs_address': '10.32.105.6:57192',
 'address': '10.32.105.6:57192',
 'node_id': 'e05f50b9e66a6960ea79c3c4f2deaab4545bc452f6d0aa437564eabc'}

In [5]:
endpoints = sorted(pd.read_csv(f"{experiment_path}/endpoints.txt", header=None)[0].to_list())

In [6]:
covariates = ["age_at_recruitment_f21022_0_0", "sex_f31_0_0",  "ethnic_background_f21000_0_0"]

In [7]:
data_covariates = pd.read_feather(f"{experiment_path}/data_covariates.feather").set_index("eid")[covariates]\
    .assign(age_at_recruitment_f21022_0_0 = lambda x: x.age_at_recruitment_f21022_0_0.astype(np.int32))

In [8]:
data_covariates_ray = ray.put(data_covariates)

In [9]:
variables_to_norm = ["age_at_recruitment_f21022_0_0"] + endpoints

In [10]:
in_path = pathlib.Path(f"{experiment_path}/loghs")
in_path.mkdir(parents=True, exist_ok=True)

out_path = f"{experiment_path}/coxph/input"
pathlib.Path(out_path).mkdir(parents=True, exist_ok=True)

In [11]:
models = models = [f.name for f in in_path.iterdir() if f.is_dir() and "ipynb_checkpoints" not in str(f)]
partitions = [i for i in range(22)] #[0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

In [12]:
from sklearn.preprocessing import StandardScaler
import pickle
import zstandard

def read_merge_data(fp_in, split, data_covariates):
    temp = pd.read_feather(f"{fp_in}/{split}.feather").set_index("eid")
    temp = temp.merge(data_covariates, left_index=True, right_index=True, how="left")
    return temp   
    
def save_pickle(data, data_path):
    with open(data_path, "wb") as fh:
        cctx = zstandard.ZstdCompressor()
        with cctx.stream_writer(fh) as compressor:
            compressor.write(pickle.dumps(data, protocol=pickle.HIGHEST_PROTOCOL))
    
@ray.remote
def norm_variables(data_covariates, model, partition, variables):

    fp_in = f"{in_path}/{model}/{partition}"
    fp_out = f"{out_path}/{model}/{partition}"
    
    if pathlib.Path(fp_in).is_dir():
        if not pathlib.Path(fp_out).is_dir():
            pathlib.Path(fp_out).mkdir(parents=True, exist_ok=True)
            for split in ["train", "valid", "test"]:
                temp = read_merge_data(fp_in, split, data_covariates)
                if split=="train": 
                    scaler = StandardScaler(with_mean=True, with_std=True, copy=True).fit(temp[variables].values)
                    save_pickle(scaler, f"{fp_out}/scaler.p")
                temp[variables] = scaler.transform(temp[variables].values)
                temp.reset_index(drop=False).to_feather(f"{fp_out}/{split}.feather")
    return True

In [13]:
#norm_variables(data_covariates, models[0], partitions[0], variables_to_norm)

In [14]:
def norm_logh_and_extra(data_covariates_ray, variables):
    progress = []
    for model in tqdm(models):
        for partition in [p for p in partitions]:
            progress.append(norm_variables.remote(data_covariates, model, partition, variables))
    [ray.get(s) for s in tqdm(progress)]

In [15]:
norm_logh_and_extra(data_covariates_ray, variables_to_norm)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/44 [00:00<?, ?it/s]

In [ ]:
1+1